# 03 Compuertas de calidad y deduplicación

Bronze conserva los mensajes. Silver exige MMSI de nueve dígitos distinto de cero, tiempo válido, coordenadas finitas en rango, posición distinta de (0, 0), agua cartografiada, SOG en [0, 60] nudos y COG en [0, 360) grados. El rumbo 360 es un centinela. Heading, eslora y manga pueden faltar. Una fila inválida se conserva con todos sus motivos.

**Por qué MMSI y BaseDateTime.** El MMSI identifica el emisor y el tiempo UTC identifica su observación. Agregar latitud o longitud a la llave retendría dos posiciones incompatibles para el mismo instante y distorsionaría la distancia en oro. Usar solo MMSI descartaría la trayectoria. Primero se evalúa cada copia para que una copia inválida no desplace a la válida. Entre candidatas válidas del mismo lote se elige de manera determinista por archivo y contenido. Si la clave ya fue aceptada en un lote anterior, se conserva la primera versión.

La máscara combina mar, lagos y cauces de Natural Earth con la línea de costa TIGER/Line 2023 y la Red Nacional de Vías Navegables de USACE. Se admite una celda que intersecte esas dos referencias estadounidenses. Ausencia de cobertura se envía a cuarentena. En la costa y en canales estrechos la cartografía puede rechazar observaciones válidas o admitir posiciones próximas al agua. Cada lote reconcilia entradas con nuevas, duplicadas dentro del lote, duplicadas anteriores y cuarentena. El nuevo checkpoint relee el histórico de Bronze y conserva el estado anterior del equipo.

In [ ]:
# Capturar los parámetros del notebook llamador antes de %run.
dbutils.widgets.text("fecha", "2023-06-08")
dbutils.widgets.text("ancla_replay", "")
dbutils.widgets.text("inicio_run", "")
ow_argumentos = tuple(dbutils.widgets.get(k).strip() for k in ("fecha", "ancla_replay", "inicio_run"))

In [ ]:
%run ./_config

In [ ]:
from pyspark.sql import Window, functions as F

COLUMNAS = ["MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG", "Heading", "VesselName",
            "IMO", "CallSign", "VesselType", "Status", "Length", "Width", "Draft", "Cargo",
            "TransceiverClass", "_source_file", "_ingested_at"]


def con_motivo(df, mascara):
    lat, lon = F.col("LAT"), F.col("LON")
    lat_ok = lat.between(-90, 90) & ~F.isnan(lat)
    lon_ok = lon.between(-180, 180) & ~F.isnan(lon)
    coord_ok = lat_ok & lon_ok
    # Índice global de píxel en orden norte-sur, oeste-este; se incluyen los extremos.
    gx = F.when(coord_ok, F.least(F.lit(71999), F.floor((lon + 180) * 200))).cast("int")
    gy = F.when(coord_ok, F.least(F.lit(35999), F.floor((90 - lat) * 200))).cast("int")
    puntos = (df.withColumn("gx", gx).withColumn("gy", gy)
              .withColumn("tesela", F.expr("int(gy div 200) * 360 + int(gx div 200)"))
              .withColumn("pixel", F.expr("pmod(gy, 200) * 200 + pmod(gx, 200)")))
    puntos = puntos.join(F.broadcast(mascara), "tesela", "left")
    # Un dígito hexadecimal representa cuatro píxeles, del bit más alto al más bajo.
    tierra = F.expr("""CASE clase WHEN 0 THEN false WHEN 1 THEN true WHEN 2 THEN
        (shiftright(cast(conv(substr(bits_hex, int(pixel div 4) + 1, 1), 16, 10) as int),
                   3 - pmod(pixel, 4)) & 1) = 1 END""")
    mmsi = F.col("MMSI")
    return puntos.select(*COLUMNAS, F.concat_ws(",",
        F.when(mmsi.isNull() | ~mmsi.rlike("^[0-9]{9}$") | (mmsi == "000000000"), "mmsi_invalido"),
        F.when(F.col("BaseDateTime").isNull(), "sin_timestamp"),
        F.when(lat.isNull() | ~lat_ok, "latitud_invalida"),
        F.when(lon.isNull() | ~lon_ok, "longitud_invalida"),
        F.when((lat == 0) & (lon == 0), "posicion_cero"),
        F.when(coord_ok & tierra, "posicion_en_tierra"),
        F.when(coord_ok & tierra.isNull(), "sin_referencia_agua"),
        F.when(F.col("SOG").isNull() | ~F.col("SOG").between(0, 60) | F.isnan("SOG"), "sog_invalido"),
        F.when(F.col("COG").isNull() | ~F.col("COG").between(0, 360) |
               (F.col("COG") == 360) | F.isnan("COG"), "cog_invalido")
    ).alias("motivo"))


def escribir_lote(df, tabla, app, lote):
    (df.write.format("delta").mode("append")
     .option("txnAppId", app).option("txnVersion", lote).saveAsTable(tabla))


def aplicar_silver(batch_df, lote):
    sesion = batch_df.sparkSession
    marcado = con_motivo(batch_df, sesion.table(MASCARA)).persist()
    candidatas = None
    try:
        tot = marcado.agg(F.count("*"), F.count(F.when(F.col("motivo") != "", 1))).first()
        entrada, rechazo = tot[0], tot[1]
        if entrada == 0:
            return
        app = "oceanwatch-v2-" + SILVER.replace("`", "")
        if rechazo:
            malas = marcado.where("motivo <> ''").withColumn("lote_calidad", F.lit(lote).cast("long"))
            escribir_lote(malas, CUARENTENA, app + "-rechazos", lote)
        validas = marcado.where("motivo = ''").drop("motivo")
        orden = Window.partitionBy("MMSI", "BaseDateTime").orderBy(
            "_source_file", F.to_json(F.struct(*COLUMNAS[:-2])), "_ingested_at")
        candidatas = (validas.withColumn("_orden", F.row_number().over(orden)).where("_orden = 1")
                      .drop("_orden").withColumn("fecha_dia", F.to_date("BaseDateTime"))
                      .withColumn("lote_alta", F.lit(lote).cast("long")).persist())
        dias = candidatas.groupBy("fecha_dia").count().collect()
        n = sum(x["count"] for x in dias)
        nuevas = 0
        if dias:
            fechas = ", ".join("DATE '" + x.fecha_dia.isoformat() + "'" for x in dias)
            candidatas.createOrReplaceTempView("ow_candidatas")
            sesion.sql(f"""MERGE INTO {SILVER} d USING ow_candidatas s
                ON d.fecha_dia IN ({fechas}) AND d.fecha_dia=s.fecha_dia
                  AND d.MMSI=s.MMSI AND d.BaseDateTime=s.BaseDateTime
                WHEN NOT MATCHED THEN INSERT *""")
            nuevas = sesion.sql(f"SELECT count(*) FROM {SILVER} WHERE fecha_dia IN ({fechas}) AND lote_alta={int(lote)}").first()[0]
        intra, previos = entrada - rechazo - n, n - nuevas
        assert min(intra, previos) >= 0 and entrada == nuevas + intra + previos + rechazo
        metricas = sesion.sql(f"""SELECT {int(lote)}L lote, DATE '{fecha}' fecha_job,
            {entrada}L entrada, {nuevas}L nuevas, {intra}L duplicados_lote,
            {previos}L duplicados_previos, {rechazo}L cuarentena, current_timestamp() ejecutado_en""")
        escribir_lote(metricas, METRICAS_V2, app + "-metricas", lote)
    finally:
        if candidatas is not None:
            candidatas.unpersist()
        marcado.unpersist()

## Ejecución y prueba de recuperación

El valor normal es `no`. Elegir `si` en una ejecución manual habilita el ensayo en tablas y checkpoint de prueba aislados.

In [ ]:
dbutils.widgets.dropdown("prueba_recuperacion", "no", ["no", "si"])
ensayo = dbutils.widgets.get("prueba_recuperacion") == "si"
fuente, checkpoint = BRONZE, CHECKPOINT_SILVER
if ensayo:
    import uuid
    from datetime import datetime

    prueba = "ow_prueba_" + uuid.uuid4().hex[:10]
    def tabla_prueba(sufijo):
        return f"`{CATALOG}`.`{SCHEMA}`.`{prueba}_{sufijo}`"

    # Las tres salidas reciben los mismos esquemas que producción, con nombres independientes.
    for original, suffix in [(SILVER, "plata"), (CUARENTENA, "rechazos"), (METRICAS_V2, "metricas")]:
        spark.sql(f"CREATE TABLE {tabla_prueba(suffix)} USING DELTA AS SELECT * FROM {original} WHERE false")
    SILVER, CUARENTENA, METRICAS_V2 = [tabla_prueba(k) for k in ("plata", "rechazos", "metricas")]
    fuente, checkpoint = tabla_prueba("bronze"), f"{CHECKPOINTS}/ensayos/{prueba}"
    ddl = spark.table(BRONZE).schema
    def prueba_fila(mmsi, lat, lon, sog):
        datos = dict(MMSI=mmsi, BaseDateTime=datetime(2023,6,8,12), LAT=lat, LON=lon,
                     SOG=sog, COG=40., _source_file="ensayo.csv", _ingested_at=datetime(2023,6,8,13))
        return tuple(datos.get(c.name) for c in ddl)
    buena = prueba_fila("123456789", 28., -85., 5.)
    filas = [buena, buena, prueba_fila("123456789", 28., -85., 102.3),
             prueba_fila("ABCDEFGHI", 28., -85., 5.), prueba_fila("987654321", 39., -101., 5.)]
    spark.createDataFrame(filas, ddl).write.format("delta").saveAsTable(fuente)


def ejecutar_stream(funcion):
    consulta = (spark.readStream.format("delta").table(fuente).writeStream.foreachBatch(funcion)
                .queryName("ow_" + ("prueba_" + prueba if ensayo else "silver_v2"))
                .option("checkpointLocation", checkpoint).trigger(availableNow=True).start())
    consulta.awaitTermination()


if ensayo:
    def fallo_despues_de_escribir(df, lote):
        aplicar_silver(df, lote)
        raise RuntimeError("OW_FALLO_CONTROLADO")
    try:
        ejecutar_stream(fallo_despues_de_escribir)
    except Exception as exc:
        if "OW_FALLO_CONTROLADO" not in str(exc):
            raise
    else:
        raise AssertionError("Se esperaba el fallo de prueba")
    ejecutar_stream(aplicar_silver)
    spark.createDataFrame([buena], ddl).write.format("delta").mode("append").saveAsTable(fuente)
    ejecutar_stream(aplicar_silver)
    lotes = spark.table(METRICAS_V2).orderBy("lote").collect()
    assert [(x.entrada,x.nuevas,x.duplicados_lote,x.duplicados_previos,x.cuarentena) for x in lotes] == [
        (5,1,1,0,3), (1,0,0,1,0)]
    assert spark.table(SILVER).count() == 1 and spark.table(CUARENTENA).count() == 3
    display(spark.table(METRICAS_V2).orderBy("lote"))
    print("Recuperación verificada en", prueba, "con el mismo checkpoint", checkpoint)
else:
    ejecutar_stream(aplicar_silver)